# 🚗 Capstone Case 2: Physical AI with a Cutebot
## Scientific Workflows for Brain and Behavioral Research
### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)

This optional capstone takes the ideas from **Notebook 07** (AI, agentic AI and physical AI) and **Lab Stage 3** (an edge device that takes commands from the cloud) and puts them on a small robot car. You will build a complete physical-AI pipeline:

**brain signal → decision → wheel speeds → safety check → a robot that moves**

The brain signal is simulated, and the control scheme is a made-up one for teaching. Real brain–machine interfaces need careful calibration for each person and are far less reliable than this.

### Two ways to do this case

| | **Path H: Hardware** | **Path S: Simulator** |
|---|---|---|
| You need | BBC micro:bit (V1 or V2), ELECFREAKS Cutebot, 3 × AAA batteries, USB cable | A web browser |
| Parts A–C | In this notebook | In this notebook |
| Part D | Your program drives the real Cutebot | The same program drives a Cutebot in the MakeCode browser simulator |

Everyone does Parts A–C here in Colab. In Part D, this notebook writes a micro:bit program for you. The **same program** runs on the real robot or in the simulator, so both paths learn the same thing.

> ⚠️ **Path H safety:** run the Cutebot on the floor, not on a table, and clear a space about 1 m × 1 m. Keep fingers and hair away from the wheels. The power switch is on the Cutebot itself.

### 🤖 Using AI Tools: Ask → Check → Build → Document

Two activities ask you to prompt an AI assistant. Before running anything it gives you, **Check** it with three questions: Does it do what I asked? Does it do anything I didn't ask for, such as downloading or deleting files or sending data anywhere? Can I explain each step in plain words? In this case the check matters more than usual, because the code ends up **moving a real object**. Each AI activity is followed by a given check cell that tests the code on cases where you already know the right answer.

> 🔒 Never paste identifiable participant data, passwords or access keys into a public AI tool.

***

## 📋 Set Up Your Profile
*Run the cell below. Choose your path from the dropdown.*

In [ ]:
# PROFILE SETUP
student_name = "Your Name"                    #@param {type:"string"}
institution = "Your University/Organization"  #@param {type:"string"}
my_path = "Path S: Simulator"                 #@param ["Path S: Simulator", "Path H: Hardware (micro:bit + Cutebot)"]

print(f"✅ Notebook initialized for {student_name} ({institution})")
print(f"Path: {my_path}")

***
## 🧠 Part A: From a Brain Signal to a Decision
### Topic: Stage 1 (Obtaining Data) and Stage 2 (Analyzing Data)

Our pretend user wears a two-channel EEG headset, one electrode over each side of the head. Every 2 seconds the system reads a **window** of signal and turns it into one of four commands: `forward`, `turn-left`, `turn-right` or `stop`.

The control scheme (made up for teaching, built from the band-power idea in Notebook 07):
- **Relaxed** (strong alpha, 8–13 Hz, on both sides) → `stop`
- **Engaged** (more beta, 13–30 Hz, than alpha) → move. Then:
  - alpha clearly weaker on the **left** → `turn-left`
  - alpha clearly weaker on the **right** → `turn-right`
  - otherwise → `forward`

### 🧭 Activity A.1 — Simulate a session and decode it

The cell below is given. It simulates a 13-window session in which the user **intends** a short route, then decodes each window with the rule above. Run it and compare what the user meant with what the decoder heard.

In [ ]:
import numpy as np
from scipy.signal import periodogram

FS = 128          # samples per second (the same rate as the GAMEEMO EEG data in Labs 1 and 2)
WINDOW_S = 2.0    # one decision every 2 seconds
NOISE = 2.0       # background noise level

# The route our pretend user intends, one command per 2-second window
INTENDED = ["forward"] * 3 + ["turn-left"] * 2 + ["forward"] * 2 + ["turn-right"] * 2 + ["forward"] * 2 + ["stop"] * 2

def simulate_window(intent, rng):
    """Stage 1: a simulated 2-second, two-channel EEG window for one intended command."""
    t = np.arange(0, WINDOW_S, 1 / FS)
    alpha_left = alpha_right = 0.6
    beta = 1.6                                   # engaged: beta stronger than alpha
    if intent == "stop":
        alpha_left = alpha_right = 2.0           # relaxed: strong alpha, weak beta
        beta = 0.6
    elif intent == "turn-left":
        alpha_left, alpha_right = 0.5, 1.4       # weaker alpha on the left
    elif intent == "turn-right":
        alpha_left, alpha_right = 1.4, 0.5      # weaker alpha on the right
    def channel(alpha):
        phases = rng.uniform(0, 2 * np.pi, 2)
        return (alpha * np.sin(2 * np.pi * 10 * t + phases[0])
                + beta * np.sin(2 * np.pi * 20 * t + phases[1])
                + rng.normal(0, NOISE, t.size))
    return channel(alpha_left), channel(alpha_right)

def band_power(signal, fs, low, high):
    """Average power between low and high Hz (the checked version from Notebook 07)."""
    freqs, power = periodogram(signal, fs=fs)
    in_band = (freqs >= low) & (freqs <= high)
    return power[in_band].mean()

ENGAGEMENT_THRESHOLD = 0.3   # beta/alpha below this counts as relaxed -> stop
ASYMMETRY_THRESHOLD = 0.3    # left-right alpha difference needed to call a turn

def decode(left, right):
    """Stage 2: turn one window into a command. Returns (command, engagement, asymmetry)."""
    a_left, a_right = band_power(left, FS, 8, 13), band_power(right, FS, 8, 13)
    b_left, b_right = band_power(left, FS, 13, 30), band_power(right, FS, 13, 30)
    engagement = (b_left + b_right) / (a_left + a_right)
    asymmetry = (a_left - a_right) / (a_left + a_right)   # negative = weaker alpha on the left
    if engagement < ENGAGEMENT_THRESHOLD:
        return "stop", engagement, asymmetry
    if asymmetry < -ASYMMETRY_THRESHOLD:
        return "turn-left", engagement, asymmetry
    if asymmetry > ASYMMETRY_THRESHOLD:
        return "turn-right", engagement, asymmetry
    return "forward", engagement, asymmetry

rng = np.random.default_rng(42)   # fixed seed: everyone gets the same session
decoded, engagements, asymmetries = [], [], []
for intent in INTENDED:
    command, eng, asym = decode(*simulate_window(intent, rng))
    decoded.append(command); engagements.append(eng); asymmetries.append(asym)

print(f"{'window':>6} | {'intended':10} | {'decoded':10} | engagement | asymmetry")
for i, (want, got) in enumerate(zip(INTENDED, decoded)):
    mark = "" if want == got else "   <-- mismatch"
    print(f"{i:>6} | {want:10} | {got:10} | {engagements[i]:10.2f} | {asymmetries[i]:9.2f}{mark}")
n_right = sum(w == g for w, g in zip(INTENDED, decoded))
print(f"\nDecoded correctly: {n_right} of {len(INTENDED)} windows ({n_right / len(INTENDED):.0%}); guessing among 4 commands: 25%")

> ✍️ **Think about it:** in Lab 2, a wrong prediction was just a lower accuracy number. Here each mismatch becomes a wrong movement. Which kind of mistake would worry you more on a real wheelchair: a `stop` heard as `forward`, or a `forward` heard as `stop`?

***
## ⚙️ Part B: From a Decision to Wheel Speeds
### Topic: The Physical-AI Boundary

A **Cutebot** is a small rear-wheel-drive car with two motors. It turns by running one wheel slower than the other. Converting a command into a pair of wheel speeds is where a decision becomes a movement: the boundary between AI that only processes data and **physical AI** (Concept A10 in Notebook 07).

### 🧭 Activity B.1 — Ask the AI for the wheel-speed mapping

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the cell beneath it.*
>
> "Write a Python function `decision_to_wheel_speeds(command)` for a two-wheel robot car. It takes one of the strings 'forward', 'turn-left', 'turn-right' or 'stop' and returns a tuple `(left_speed, right_speed)`, each a whole number from -100 to 100 (percent of full power). Use 50 for forward on both wheels. For a turn, keep the outer wheel at 50 and slow the inner wheel to 10. 'stop' returns (0, 0). **Any other input, including misspellings or 'ask human', must also return (0, 0)**, because an unknown command should never move a robot. Add a comment for each case explaining which way the car moves."

In [ ]:
# 💻 Paste your AI-generated code here, then run this cell.

**✅ Check before you trust it.** The cell below is given. It tests your function on cases where we already know the right answer. Run it. If any line fails, ask the AI to fix that case, paste the fixed version above, and run both cells again.

In [ ]:
# Given known-answer check for decision_to_wheel_speeds
checks = [
    ("forward",    lambda s: s[0] == s[1] and s[0] > 0,      "both wheels equal and moving forward"),
    ("turn-left",  lambda s: 0 <= s[0] < s[1],               "left wheel slower than right"),
    ("turn-right", lambda s: 0 <= s[1] < s[0],               "right wheel slower than left"),
    ("stop",       lambda s: s == (0, 0),                    "both wheels off"),
    ("ask human",  lambda s: s == (0, 0),                    "an unknown command stops the car"),
    ("forwrad",    lambda s: s == (0, 0),                    "a misspelled command stops the car"),
]
all_ok = True
for command, test, meaning in checks:
    try:
        speeds = tuple(decision_to_wheel_speeds(command))
        ok = test(speeds) and all(-100 <= v <= 100 for v in speeds)
    except Exception as err:
        speeds, ok = f"error: {err}", False
    all_ok &= ok
    print(f"{'✅' if ok else '❌'} {command!r:13} -> {speeds}   ({meaning})")
print("\nAll checks passed." if all_ok else "\nSome checks failed: fix them before going on. This code will move a robot.")

### 🧭 Activity B.2 — Where would the car actually go?

The cell below is given. It uses a simple model of the Cutebot (wheel speed → distance per second; turning from the speed difference between the wheels) to draw the path the car **would** drive for the intended route and for the decoded route.

In [ ]:
import matplotlib.pyplot as plt

STEP_S = 0.8            # each command runs for 0.8 seconds
CM_PER_S_AT_100 = 20.0  # model: full power moves the car about 20 cm per second (a teaching value, not a measured one)
WHEEL_BASE_CM = 9.0     # model: distance between the two wheels

def drive_path(commands, wheel_fn, start=(0.0, 0.0, np.pi / 2), safety=None, ask=None):
    """Simulate the car. Returns the list of (x, y) points and the commands actually driven."""
    x, y, heading = start
    points, driven = [(x, y)], []
    for i, command in enumerate(commands):
        if ask is not None:
            command = ask(i, command)                  # optional human-in-the-loop (Part C)
        if safety is not None:
            command = safety(command, x, y, heading)   # optional edge safety rule (Part C)
        driven.append(command)
        left, right = wheel_fn(command)
        v_left, v_right = left / 100 * CM_PER_S_AT_100, right / 100 * CM_PER_S_AT_100
        for _ in range(20):                            # small time steps for a smooth path
            dt = STEP_S / 20
            speed, turn_rate = (v_left + v_right) / 2, (v_right - v_left) / WHEEL_BASE_CM
            x += speed * np.cos(heading) * dt
            y += speed * np.sin(heading) * dt
            heading += turn_rate * dt
            points.append((x, y))
    return np.array(points), driven

intended_path, _ = drive_path(INTENDED, decision_to_wheel_speeds)
decoded_path, _ = drive_path(decoded, decision_to_wheel_speeds)

plt.figure(figsize=(6, 6))
plt.plot(*intended_path.T, "g-", lw=3, alpha=0.6, label="intended route")
plt.plot(*decoded_path.T, "b--", lw=2, label="decoded route")
plt.plot(0, 0, "ko", label="start")
plt.axis("equal"); plt.xlabel("x (cm)"); plt.ylabel("y (cm)")
plt.title("Same user, same intention: two different journeys"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

gap = np.linalg.norm(intended_path[-1] - decoded_path[-1])
print(f"The decoded route ends {gap:.0f} cm away from where the user meant to go.")

***
## 🛡️ Part C: Safety and Human Oversight
### Topic: Who Has the Last Word?

The decoder can be wrong, so a well-designed physical-AI system adds layers that can **overrule** it. We'll add two, matching Notebook 07's autonomy spectrum:

1. **An edge safety rule** on the robot itself: if the ultrasonic distance sensor sees something closer than 15 cm, stop, whatever the decoder said. It runs on the device, so it works even if the connection to the cloud drops.
2. **A human in the loop** for decisions the decoder was unsure about: the robot pauses and a person confirms or skips the step.

### 🧭 Activity C.1 — An edge safety rule

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the cell beneath it.*
>
> "Write a Python function `edge_safety(command, distance_cm, safe_cm=15)`. It returns 'stop' if `distance_cm` is less than `safe_cm` and the command would move the car (anything other than 'stop'); otherwise it returns the command unchanged. Treat a `distance_cm` of None as 'no reading' and return 'stop' to be safe. Add comments explaining why this rule belongs on the robot rather than in the cloud."

In [ ]:
# 💻 Paste your AI-generated code here, then run this cell.

**✅ Check it**, then run the car toward a wall. The cell is given.

In [ ]:
# Given known-answer check for edge_safety
cases = [(("forward", 50), "forward"), (("forward", 10), "stop"), (("turn-left", 14.9), "stop"),
         (("stop", 5), "stop"), (("forward", None), "stop"), (("turn-right", 15), "turn-right")]
ok_all = True
for (command, dist), expected in cases:
    got = edge_safety(command, dist)
    ok_all &= got == expected
    print(f"{'✅' if got == expected else '❌'} edge_safety({command!r}, {dist}) -> {got!r} (expected {expected!r})")
print("\nAll checks passed." if ok_all else "\nFix the failing case before going on.")

# A wall 40 cm straight ahead of the start. The car only "sees" it when facing roughly toward it.
WALL_Y = 40.0
def distance_to_wall(x, y, heading):
    facing = np.sin(heading)
    return (WALL_Y - y) / facing if facing > 0.3 else 999.0

def safety_layer(command, x, y, heading):
    return edge_safety(command, distance_to_wall(x, y, heading))

straight = ["forward"] * 12
no_safety, _ = drive_path(straight, decision_to_wheel_speeds)
with_safety, driven = drive_path(straight, decision_to_wheel_speeds, safety=safety_layer)

plt.figure(figsize=(5, 5))
plt.axhline(WALL_Y, color="k", lw=4, label="wall")
plt.plot(*no_safety.T, "r--", label="no safety rule")
plt.plot(*with_safety.T, "g-", lw=3, label="with edge safety rule")
plt.axis("equal"); plt.ylim(-5, 110); plt.xlabel("x (cm)"); plt.ylabel("y (cm)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Twelve 'forward' commands toward a wall"); plt.show()
print("Commands actually driven with the safety rule:", driven)

### 🧭 Activity C.2 — A human in the loop for unsure decisions

A decision is "unsure" when its numbers sit close to a threshold. The cell below is given. It flags any window whose engagement or asymmetry is within 0.1 of a threshold (except `stop`, which is always safe), and simulates a person who, when asked, knows what they meant: they confirm a correct step and skip a wrong one.

In [ ]:
MARGIN = 0.1

def is_unsure(eng, asym):
    near_engagement = abs(eng - ENGAGEMENT_THRESHOLD) < MARGIN
    near_turn = eng >= ENGAGEMENT_THRESHOLD and abs(abs(asym) - ASYMMETRY_THRESHOLD) < MARGIN
    return near_engagement or near_turn

# Stopping is always safe, so a 'stop' never needs approval
unsure = [is_unsure(e, a) and d != "stop" for e, a, d in zip(engagements, asymmetries, decoded)]

def human_check(i, command):
    if not unsure[i]:
        return command                                     # confident: no interruption
    verdict = "confirms" if command == INTENDED[i] else "skips"
    print(f"  window {i}: decoder unsure about {command!r} -> human {verdict} it")
    return command if command == INTENDED[i] else "stop"   # a skipped step does nothing

print("Asking the human:")
hitl_path, hitl_driven = drive_path(decoded, decision_to_wheel_speeds, ask=human_check)

errors_before = sum(w != g for w, g in zip(INTENDED, decoded))
errors_after = sum(w != d and d != "stop" for w, d in zip(INTENDED, hitl_driven))
print(f"\nWrong movements without the human: {errors_before}. With the human: {errors_after}. Interruptions: {sum(unsure)}.")
print("Errors the human never saw (the decoder was confident and wrong):",
      [i for i, (w, g) in enumerate(zip(INTENDED, decoded)) if w != g and not unsure[i]])

plt.figure(figsize=(6, 6))
plt.plot(*intended_path.T, "g-", lw=3, alpha=0.6, label="intended route")
plt.plot(*decoded_path.T, "b--", lw=2, label="decoded, no human")
plt.plot(*hitl_path.T, "m-", lw=2, label="decoded, human checks unsure steps")
plt.axis("equal"); plt.xlabel("x (cm)"); plt.ylabel("y (cm)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

### ✍️ Part B–C Reflection
*Double-click this cell to write your response.*

* **What I observed:** How far did the decoded route end from the intended one? Which errors did the human catch, and why were the others missed?
* **Checking AI code:** Did the AI's first version of `decision_to_wheel_speeds` or `edge_safety` pass every check? If not, what did it get wrong?
* **Who has the last word?** List the three layers that can decide what the car does (decoder, edge safety rule, human). Which one should win when they disagree, and why?
* **Connecting to key concepts:** Notebook 07 Concepts **A9** (perception–decision–action loop), **A10** (physical AI), **A11** (autonomy spectrum), **A12** (human-in-the-loop) and **A13** (failure modes).

***
## 🤖 Part D: Run It on a Robot
### Topic: Cloud Decides, Edge Acts (Stage 4: Documenting Research follows)

The cell below turns your decoded commands into a **micro:bit program** written in MakeCode JavaScript. It uses *your* checked `decision_to_wheel_speeds` for the wheel speeds, and it carries over both safety layers from Part C:

- **Edge safety rule:** before every step the robot reads its ultrasonic sensor and stops if anything is closer than 15 cm. Its headlights turn red.
- **Human in the loop:** steps the decoder was unsure about are marked `ask:`. The robot stops, its headlights turn yellow, and it waits for you. **Button A** = do it; **A and B together** = skip it.
- **Button A** also starts the run. **Button B** is an emergency stop at any time.

The program uses Microsoft's free **micro:bit robot** extension for MakeCode, which supports the ELECFREAKS Cutebot and includes a robot simulator. Run the cell, then follow the steps for your path below it.

In [ ]:
from datetime import date

def make_makecode_program(commands, unsure_flags, wheel_fn, step_ms=800, safe_cm=15):
    """Write a MakeCode JavaScript program that drives a Cutebot through `commands`."""
    steps = [("ask:" + c) if flag and c != "stop" else c for c, flag in zip(commands, unsure_flags)]
    speeds = {c: wheel_fn(c) for c in ["forward", "turn-left", "turn-right"]}
    command_list = ", ".join(f'"{s}"' for s in steps)
    return f'''// Capstone Case 2 (CI Workshop): generated in Colab on {date.today().isoformat()}
// Paste into the JavaScript tab of a MakeCode project that has the extension
// https://github.com/microsoft/microbit-robot
robot.elecfreaksCuteBot.start()
let commands = [{command_list}]
let STEP_MS = {step_ms}
let SAFE_CM = {safe_cm}
let step = 0
let running = false
let waiting = false
let confirmed = false
let skipped = false

function drive(command: string) {{
    if (command == "forward") {{
        robot.motorTank({speeds["forward"][0]}, {speeds["forward"][1]})
    }} else if (command == "turn-left") {{
        robot.motorTank({speeds["turn-left"][0]}, {speeds["turn-left"][1]})
    }} else if (command == "turn-right") {{
        robot.motorTank({speeds["turn-right"][0]}, {speeds["turn-right"][1]})
    }} else {{
        robot.motorStop()
    }}
}}

// Button A: start the run, or approve a step the decoder was unsure about
input.onButtonPressed(Button.A, function () {{
    if (!running) {{
        running = true
        step = 0
    }} else if (waiting) {{
        confirmed = true
    }}
}})
// Buttons A+B together: skip an unsure step
input.onButtonPressed(Button.AB, function () {{
    if (waiting) {{
        skipped = true
    }}
}})
// Button B: emergency stop, always available
input.onButtonPressed(Button.B, function () {{
    running = false
    waiting = false
    robot.motorStop()
    robot.setColor(0xff0000)
}})

basic.forever(function () {{
    if (!running) {{
        return
    }}
    if (step >= commands.length) {{
        robot.motorStop()
        robot.setColor(0x00ff00)
        running = false
        return
    }}
    let command = commands[step]
    if (command.indexOf("ask:") == 0) {{
        if (!confirmed && !skipped) {{
            robot.motorStop()
            robot.setColor(0xffff00)
            waiting = true
            return
        }}
        command = skipped ? "stop" : command.substr(4)
        confirmed = false
        skipped = false
        waiting = false
    }}
    // Edge safety rule: the robot has the last word, whatever the cloud decided
    let distance = robot.obstacleDistance()
    if (distance > 0 && distance < SAFE_CM && command != "stop") {{
        robot.motorStop()
        robot.setColor(0xff0000)
    }} else {{
        robot.setColor(0x0000ff)
        drive(command)
    }}
    basic.pause(STEP_MS)
    step += 1
}})
'''

program = make_makecode_program(decoded, unsure, decision_to_wheel_speeds)
with open("cutebot_program.js", "w") as f:
    f.write(program)
print(program)
print("Saved as cutebot_program.js (Files panel on the left). Copy everything above from the first // line.")

### Path S: Simulator (browser only)

1. Open **[MakeCode for micro:bit](https://makecode.microbit.org)**{target="_blank"} and click **New Project**. Give it a name.
2. Click the ⚙️ gear (top right) → **Extensions**. Paste `https://github.com/microsoft/microbit-robot` into the search box and press Enter. Click the **microbit-robot** card that appears. A new orange **Robot** category appears in the toolbox.
3. Click **JavaScript** at the top of the editor. Select everything in the editor and delete it, then paste the program from the cell above.
4. If the simulator shows a ▶ play button, click it. A robot on an oval track appears under the micro:bit. Press **A** on the simulated micro:bit to start.
5. Watch the car: blue lights = driving, yellow = waiting for you (press **A** to approve, or **A+B** to skip), red = stopped, green = finished. You can drag the robot with the mouse to reset it.
6. Optional: the simulated track has two small blocks in the middle. Drag the robot so it faces one of them, press **A** again, and see whether the safety rule stops it.

### Path H: Hardware (micro:bit + Cutebot)

1. Do steps 1–3 of Path S.
2. Connect the micro:bit to your computer with the USB cable and click **Download**. If MakeCode offers to pair your micro:bit, follow its steps; otherwise copy the downloaded `.hex` file onto the **MICROBIT** drive.
3. Unplug the micro:bit and push it into the Cutebot's slot with the LED screen facing forward. Check the 3 AAA batteries, put the car on the floor in a clear space, and switch the Cutebot on.
4. Press **A** to start. Keep a hand ready for **B** (emergency stop).
5. Hold a hand or a book about 10 cm in front of the car while it drives forward: the edge safety rule should stop it and turn the lights red.

> 💡 **If you already use the ELECFREAKS "Cutebot" extension**, start a new project for this case instead of adding both extensions to one project.

### 🧭 Activity D.1 — Document the run (Stage 4)

Run the cell below and fill in what happened. It saves a small record of the run next to the program, so someone else could reproduce it.

In [ ]:
import json

robot_followed_route = "Mostly"     #@param ["Yes", "Mostly", "No", "Did not run it"]
safety_stop_seen = "Not tested"     #@param ["Yes", "No", "Not tested"]
times_asked_to_approve = 0          #@param {type:"integer"}
what_surprised_me = "One sentence about what was different on the robot or simulator from the plots above."  #@param {type:"string"}

run_record = {
    "participant": student_name,
    "path": my_path,
    "date": date.today().isoformat(),
    "decoder": {"engagement_threshold": ENGAGEMENT_THRESHOLD, "asymmetry_threshold": ASYMMETRY_THRESHOLD,
                "noise": NOISE, "seed": 42, "windows_correct": int(n_right), "windows_total": len(INTENDED)},
    "wheel_speeds": {c: list(decision_to_wheel_speeds(c)) for c in ["forward", "turn-left", "turn-right", "stop"]},
    "commands_sent": [("ask:" + c) if f and c != "stop" else c for c, f in zip(decoded, unsure)],
    "robot_followed_route": robot_followed_route,
    "safety_stop_seen": safety_stop_seen,
    "times_asked_to_approve": times_asked_to_approve,
    "what_surprised_me": what_surprised_me,
}
with open("cutebot_run_record.json", "w") as f:
    json.dump(run_record, f, indent=2)
print(json.dumps(run_record, indent=2))
print("\nSaved as cutebot_run_record.json. Download it with cutebot_program.js before you close Colab (the workspace is temporary).")

***
## 📈 Part E (Stretch): A Learned Decoder
### Topic: Rule-Based vs. Learned Control (Notebook 07, Concept A4)

The thresholds in Part A were picked by hand. A learned decoder instead fits the boundaries from a **calibration session**: the user deliberately thinks each command several times while the system records labelled windows. The fair test is on a *different* session, never on the calibration data (the same leakage lesson as Lab 2).

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the cell beneath it.*
>
> "I have a function `simulate_window(intent, rng)` that returns two EEG channels for one of the commands 'forward', 'turn-left', 'turn-right' or 'stop', and a function `band_power(signal, fs, low, high)`. FS is a global. Write Python that (1) builds a calibration set of 40 windows per command with `rng = np.random.default_rng(7)`, using four features per window: the log of alpha (8–13 Hz) and beta (13–30 Hz) power on each channel; (2) trains a scikit-learn `LogisticRegression(max_iter=1000)` on it; (3) builds a separate test session of 40 windows per command with `np.random.default_rng(99)`; (4) prints the test accuracy of the trained model and of my existing rule `decode(left, right)` (which returns the command as its first value) on the same test windows, next to the 25% guessing level."

In [ ]:
# 💻 Paste your AI-generated code here, then run this cell.

> ✍️ **Stretch reflection:** which decoder would you put on a robot that moves near people, and what would you need to see before trusting the other one?

***
## 📑 Where This Leads

You built the full loop that the 4760 course and current physical-AI research build on: sense a signal, decide, act, and keep a human and a safety rule able to overrule the AI. Natural next steps, all optional:
- Swap the made-up control scheme for real EEG features from the Lab 1 and Lab 2 GAMEEMO data.
- Let the robot's own sensors close the loop, for example the Cutebot's line sensors (try the MakeCode robot extension's **line follower** tutorial).
- Read **[Where the Field Is Headed](https://019f883f-d93e-21a6-f811-ae2939c5175d.share.connect.posit.cloud/Where_The_Field_Is_Headed.html)**{target="_blank"} for where agentic AI for science is going.

***
## ✅ Self-Check: What Did You Learn?

*Uses the Notebook 07 Reference Guide (Concepts A1–A15).*

| Concept Area | Concept | Where You Demonstrated It |
|---|---|---|
| What AI is | A4 — Rule-based vs. learned control | [ Enter Cell ID ] |
| What AI is | A7 — Known-answer test | [ Enter Cell ID ] |
| Agentic AI | A9 — Perception–decision–action loop | [ Enter Cell ID ] |
| Agentic AI | A10 — Physical AI | [ Enter Cell ID ] |
| Agentic AI | A12 — Human-in-the-loop | [ Enter Cell ID ] |
| Agentic AI | A13 — Failure modes | [ Enter Cell ID ] |

***
*End of Capstone Case 2 — Physical AI with a Cutebot*